# Transformer Block Coupling and Cross-Token Influence Metrics
### Google Colab Version

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugolov/coupling/blob/main/metrics_demo_colab.ipynb)

This notebook demonstrates the computation and visualization of:
1. **Block Coupling Metrics** - measuring alignment between layer Jacobians (ICLR 2025)
2. **Cross-Token Influence Metrics** - measuring how each input token influences the output

## Mathematical Background

### Block Coupling
For residual blocks $X^{l+1} = X^l + f^{l+1}(X^l)$, we compute:
- Jacobians: $J^l = \frac{\partial f^l(X^{l-1})}{\partial X^{l-1}}$
- SVD: $J_1 = U_1 S_1 V_1^T$, $J_2 = U_2 S_2 V_2^T$
- Coupling: $m_K(J_1, J_2) = \frac{||U_2^T J_1 V_2 - S_1||_F}{||s_1||_p}$

### Cross-Token Influence
For each layer $l$ and input token $j$, compute residual Jacobian:
$$J_{residual} = \frac{\partial f^l(X^{l-1})_T}{\partial x_j^{l-1}}$$

Then measure via:
- Frobenius norm: $||J||_F = \sqrt{\sum S^2}$
- Spectral norm: $\max(S)$
- Participation ratio: $(\sum S)^2 / \sum S^2$
- Entropy effective rank: $\exp(-\sum p_i \log p_i)$ where $p_i = s_i / \sum s_i$

---

⚠️ **Requirements**: This notebook requires a **GPU runtime**. Go to Runtime → Change runtime type → Select GPU (T4 recommended).

## 1. Setup and Installation

In [ ]:
# Check GPU availability
import torch

if torch.cuda.is_available():
    print("✓ GPU is available!")
    print(f"  Device: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("✗ GPU not available!")
    print("⚠️ This notebook requires a GPU. Please enable it:")
    print("   Runtime → Change runtime type → Hardware accelerator → GPU")
    raise RuntimeError("GPU required but not available")

In [ ]:
# Install the coupling package and dependencies
print("Installing coupling package from GitHub...\n")
!pip install -q git+https://github.com/sugolov/coupling.git

print("\n✓ Installation complete!")

In [ ]:
# Import required libraries
import os
import torch
import matplotlib.pyplot as plt
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from coupling import (
    run_coupling_hf,
    run_cross_token_influence_hf,
    coupling_from_hooks,
    cross_token_influence_from_hooks
)

# Set plotting style
plt.style.use('default')
%matplotlib inline

print("✓ All libraries imported successfully!")

### Optional: Mount Google Drive (for saving results)

In [ ]:
# Uncomment to mount Google Drive for saving results
# from google.colab import drive
# drive.mount('/content/drive')
# save_dir = '/content/drive/MyDrive/coupling_results'
# os.makedirs(save_dir, exist_ok=True)
# print(f"Results will be saved to: {save_dir}")

### Optional: HuggingFace Authentication (for gated models like Llama)

In [ ]:
# Uncomment and add your HuggingFace token to access gated models
# This is required for models like meta-llama/Meta-Llama-3-8B
# Get your token from: https://huggingface.co/settings/tokens

# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN_HERE")

# Or use Colab secrets (recommended):
# from google.colab import userdata
# login(token=userdata.get('HF_TOKEN'))

## 2. Load Model and Tokenizer

We'll use 4-bit quantization to fit larger models in Colab's GPU memory.

**Recommended models for T4 GPU (15GB):**
- `meta-llama/Meta-Llama-3-8B` (requires HF authentication)
- `gpt2` or `gpt2-medium` (no authentication needed, good for testing)
- `facebook/opt-1.3b` (smaller, faster)
- `EleutherAI/pythia-1.4b` (open access)

In [ ]:
# Model configuration
# Change this to your preferred model
model_path = "gpt2"  # Start with gpt2 for testing, then try larger models
# model_path = "meta-llama/Meta-Llama-3-8B"  # Requires HF auth

model_name = os.path.normpath(os.path.basename(model_path))

# Load model with 4-bit quantization to save memory
bnb_config = BitsAndBytesConfig(load_in_4bit=True)

print(f"Loading model: {model_path}")
print("This may take a few minutes...\n")

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="cuda",
    trust_remote_code=True,
    quantization_config=bnb_config
)

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    use_fast=True,
)

print(f"✓ Model loaded successfully!")
print(f"  Model name: {model_name}")
print(f"  Number of layers: {model.config.num_hidden_layers}")
print(f"  Hidden size: {model.config.hidden_size}")

# Check GPU memory usage
print(f"\nGPU Memory Usage:")
print(f"  Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
print(f"  Reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

## 3. Define Test Prompts

We'll test with a few different prompts to see how metrics vary.

In [ ]:
prompts = [
    "What is the capital of France? The capital is",
    "The quick brown fox jumps over the lazy",
    "To be or not to be, that is the"
]

# Display tokenized prompts
print("Prompts and their tokens:\n")
for i, prompt in enumerate(prompts):
    tokens = tokenizer(prompt, return_tensors='pt')
    token_ids = tokens.input_ids[0]
    token_strs = [tokenizer.decode([tid]) for tid in token_ids]
    print(f"Prompt {i}: {prompt}")
    print(f"  Tokens ({len(token_ids)}): {token_strs}")
    print()

## 4. Block Coupling Metrics

Compute coupling between all pairs of layers. This shows how the singular vector structure of one layer's Jacobian aligns with another's.

⏱️ **Note**: This computation may take 5-15 minutes depending on the model size and number of prompts.

In [ ]:
# Compute block coupling
print("Computing block coupling metrics...")
print("This may take several minutes...\n")

coupling_results = run_coupling_hf(
    model, 
    tokenizer, 
    model_name, 
    prompts, 
    save=False, 
    verbose=True
)

print("\n✓ Block coupling computation complete!")

### Visualize Block Coupling Matrices

The coupling matrix shows how each layer pair is coupled. Lower values indicate stronger alignment.

In [ ]:
def plot_coupling_matrix(coupling_dict, K=10, prompt_idx=0, variant='ujv'):
    """
    Plot the coupling matrix as a heatmap.
    
    Args:
        coupling_dict: Results from run_coupling_hf
        K: Number of singular vectors (10, 30, or 50)
        prompt_idx: Which prompt to visualize
        variant: 'ujv' or 'vju'
    """
    key = f"coupling_{variant}"
    
    # Extract coupling matrix (using norm-normalized version)
    coupling_matrix = coupling_dict[prompt_idx][key][K]['norm'].cpu().numpy()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot full matrix
    im1 = ax1.imshow(coupling_matrix, cmap='viridis', aspect='auto')
    ax1.set_xlabel('Layer j', fontsize=12)
    ax1.set_ylabel('Layer i', fontsize=12)
    ax1.set_title(f'Block Coupling Matrix ({variant.upper()}, K={K})\n' + 
                  f'Prompt: "{coupling_dict[prompt_idx]["prompt"][:50]}..."', 
                  fontsize=14)
    plt.colorbar(im1, ax=ax1, label='Coupling (lower = more aligned)')
    
    # Plot diagonal and off-diagonal separately
    num_layers = coupling_matrix.shape[0]
    layers = np.arange(num_layers)
    
    # Diagonal (self-coupling)
    diagonal = np.diag(coupling_matrix)
    ax2.plot(layers, diagonal, 'o-', label='Self-coupling (diagonal)', linewidth=2)
    
    # Mean off-diagonal per layer
    off_diag_mean = []
    for i in range(num_layers):
        mask = np.ones(num_layers, dtype=bool)
        mask[i] = False
        off_diag_mean.append(coupling_matrix[i, mask].mean())
    
    ax2.plot(layers, off_diag_mean, 's-', label='Mean cross-layer coupling', linewidth=2, alpha=0.7)
    ax2.set_xlabel('Layer', fontsize=12)
    ax2.set_ylabel('Coupling', fontsize=12)
    ax2.set_title('Coupling vs Layer Depth', fontsize=14)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\nCoupling Statistics (K={K}, {variant.upper()}):")
    print(f"  Min coupling: {coupling_matrix.min():.4f}")
    print(f"  Max coupling: {coupling_matrix.max():.4f}")
    print(f"  Mean coupling: {coupling_matrix.mean():.4f}")
    print(f"  Mean diagonal: {diagonal.mean():.4f}")
    print(f"  Mean off-diagonal: {np.mean(off_diag_mean):.4f}")

# Plot for first prompt
plot_coupling_matrix(coupling_results, K=10, prompt_idx=0, variant='ujv')

### Compare Different K Values

The number of singular vectors K affects the coupling measurement.

In [ ]:
def compare_K_values(coupling_dict, prompt_idx=0, variant='ujv'):
    """
    Compare coupling matrices for different K values.
    """
    K_values = [10, 30, 50]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for idx, K in enumerate(K_values):
        coupling_matrix = coupling_dict[prompt_idx][f"coupling_{variant}"][K]['norm'].cpu().numpy()
        
        im = axes[idx].imshow(coupling_matrix, cmap='viridis', aspect='auto')
        axes[idx].set_xlabel('Layer j', fontsize=11)
        axes[idx].set_ylabel('Layer i', fontsize=11)
        axes[idx].set_title(f'K={K}\nMean: {coupling_matrix.mean():.4f}', fontsize=12)
        plt.colorbar(im, ax=axes[idx])
    
    fig.suptitle(f'Coupling Matrices for Different K Values ({variant.upper()})', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

compare_K_values(coupling_results, prompt_idx=0, variant='ujv')

## 5. Cross-Token Influence Metrics

Compute how each input token influences the final output token across all layers.

⏱️ **Note**: This computation may take 10-30 minutes depending on the model size and sequence length.

In [ ]:
# Compute cross-token influence
print("Computing cross-token influence metrics...")
print("This may take several minutes...\n")

influence_results = run_cross_token_influence_hf(
    model,
    tokenizer,
    model_name,
    prompts,
    save=False,
    verbose=True
)

print("\n✓ Cross-token influence computation complete!")

### Visualize Cross-Token Influence

Each metric shows different aspects of how tokens influence the output.

In [ ]:
def plot_cross_token_influence(influence_dict, prompt_idx=0):
    """
    Plot all four cross-token influence metrics.
    """
    prompt = influence_dict[prompt_idx]['prompt']
    
    # Get token strings
    tokens = tokenizer(prompt, return_tensors='pt')
    token_ids = tokens.input_ids[0]
    token_strs = [tokenizer.decode([tid]) for tid in token_ids]
    
    metrics = {
        'frobenius_norm': 'Frobenius Norm',
        'spectral_norm': 'Spectral Norm',
        'participation_ratio': 'Participation Ratio',
        'entropy_effective_rank': 'Entropy Effective Rank'
    }
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    axes = axes.flatten()
    
    for idx, (metric_key, metric_name) in enumerate(metrics.items()):
        data = influence_dict[prompt_idx][metric_key].cpu().numpy()
        
        # Plot heatmap
        im = axes[idx].imshow(data, cmap='viridis', aspect='auto')
        axes[idx].set_xlabel('Input Token Position', fontsize=11)
        axes[idx].set_ylabel('Layer', fontsize=11)
        axes[idx].set_title(f'{metric_name}\nHigher = stronger influence', fontsize=12)
        
        # Add token labels on x-axis
        axes[idx].set_xticks(range(len(token_strs)))
        axes[idx].set_xticklabels(token_strs, rotation=45, ha='right', fontsize=9)
        
        plt.colorbar(im, ax=axes[idx])
    
    fig.suptitle(f'Cross-Token Influence: "{prompt[:60]}..."', fontsize=14, y=1.00)
    plt.tight_layout()
    plt.show()

plot_cross_token_influence(influence_results, prompt_idx=0)

### Analyze Influence Patterns Across Layers

In [ ]:
def plot_influence_by_layer(influence_dict, prompt_idx=0):
    """
    Plot how influence from different tokens changes across layers.
    """
    prompt = influence_dict[prompt_idx]['prompt']
    tokens = tokenizer(prompt, return_tensors='pt')
    token_strs = [tokenizer.decode([tid]) for tid in tokens.input_ids[0]]
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    axes = axes.flatten()
    
    metrics = {
        'frobenius_norm': 'Frobenius Norm',
        'spectral_norm': 'Spectral Norm',
        'participation_ratio': 'Participation Ratio',
        'entropy_effective_rank': 'Entropy Effective Rank'
    }
    
    for idx, (metric_key, metric_name) in enumerate(metrics.items()):
        data = influence_dict[prompt_idx][metric_key].cpu().numpy()
        num_layers, num_tokens = data.shape
        
        # Plot each token's influence across layers
        for token_idx in range(num_tokens):
            axes[idx].plot(
                range(num_layers),
                data[:, token_idx],
                label=f'Token {token_idx}: {token_strs[token_idx][:10]}',
                marker='o' if num_tokens <= 5 else None,
                markersize=3,
                linewidth=2,
                alpha=0.7
            )
        
        axes[idx].set_xlabel('Layer', fontsize=11)
        axes[idx].set_ylabel(metric_name, fontsize=11)
        axes[idx].set_title(f'{metric_name} vs Layer Depth', fontsize=12)
        axes[idx].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        axes[idx].grid(True, alpha=0.3)
    
    fig.suptitle(f'Token Influence Across Layers: "{prompt[:60]}..."', fontsize=14)
    plt.tight_layout()
    plt.show()

plot_influence_by_layer(influence_results, prompt_idx=0)

### Token Importance Analysis

In [ ]:
def analyze_token_importance(influence_dict, prompt_idx=0):
    """
    Analyze which tokens have the most influence on average.
    """
    prompt = influence_dict[prompt_idx]['prompt']
    tokens = tokenizer(prompt, return_tensors='pt')
    token_strs = [tokenizer.decode([tid]) for tid in tokens.input_ids[0]]
    
    # Average across all layers for each token
    metrics = ['frobenius_norm', 'spectral_norm', 'participation_ratio', 'entropy_effective_rank']
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()
    
    metric_names = {
        'frobenius_norm': 'Frobenius Norm',
        'spectral_norm': 'Spectral Norm',
        'participation_ratio': 'Participation Ratio',
        'entropy_effective_rank': 'Entropy Effective Rank'
    }
    
    for idx, metric_key in enumerate(metrics):
        data = influence_dict[prompt_idx][metric_key].cpu().numpy()
        
        # Mean influence across layers for each token
        mean_influence = data.mean(axis=0)
        std_influence = data.std(axis=0)
        
        x_pos = np.arange(len(token_strs))
        axes[idx].bar(x_pos, mean_influence, yerr=std_influence, 
                     capsize=5, alpha=0.7, color='steelblue')
        axes[idx].set_xlabel('Token', fontsize=11)
        axes[idx].set_ylabel(f'Mean {metric_names[metric_key]}', fontsize=11)
        axes[idx].set_title(f'Average {metric_names[metric_key]} per Token\n(across all layers)', 
                           fontsize=12)
        axes[idx].set_xticks(x_pos)
        axes[idx].set_xticklabels(token_strs, rotation=45, ha='right', fontsize=9)
        axes[idx].grid(True, alpha=0.3, axis='y')
    
    fig.suptitle(f'Token Importance Analysis: "{prompt[:60]}..."', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Print most influential tokens
    print("\nMost Influential Tokens (by Frobenius norm):")
    frob_data = influence_dict[prompt_idx]['frobenius_norm'].cpu().numpy()
    mean_frob = frob_data.mean(axis=0)
    sorted_indices = np.argsort(mean_frob)[::-1]
    for i, idx in enumerate(sorted_indices[:min(5, len(token_strs))]):
        print(f"  {i+1}. Token {idx} ('{token_strs[idx]}'): {mean_frob[idx]:.4f}")

analyze_token_importance(influence_results, prompt_idx=0)

### Layer-wise Statistics

In [ ]:
def analyze_layer_statistics(influence_dict, prompt_idx=0):
    """
    Analyze how influence statistics change across layers.
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()
    
    metrics = ['frobenius_norm', 'spectral_norm', 'participation_ratio', 'entropy_effective_rank']
    metric_names = {
        'frobenius_norm': 'Frobenius Norm',
        'spectral_norm': 'Spectral Norm',
        'participation_ratio': 'Participation Ratio',
        'entropy_effective_rank': 'Entropy Effective Rank'
    }
    
    for idx, metric_key in enumerate(metrics):
        data = influence_dict[prompt_idx][metric_key].cpu().numpy()
        num_layers = data.shape[0]
        
        # Statistics per layer
        layer_mean = data.mean(axis=1)
        layer_std = data.std(axis=1)
        layer_max = data.max(axis=1)
        layer_min = data.min(axis=1)
        
        layers = np.arange(num_layers)
        
        # Plot mean with std band
        axes[idx].plot(layers, layer_mean, 'o-', label='Mean', linewidth=2, color='steelblue')
        axes[idx].fill_between(layers, layer_mean - layer_std, layer_mean + layer_std, 
                              alpha=0.3, color='steelblue', label='±1 std')
        axes[idx].plot(layers, layer_max, 's--', label='Max', linewidth=1.5, 
                      alpha=0.6, color='darkgreen')
        axes[idx].plot(layers, layer_min, '^--', label='Min', linewidth=1.5, 
                      alpha=0.6, color='darkred')
        
        axes[idx].set_xlabel('Layer', fontsize=11)
        axes[idx].set_ylabel(metric_names[metric_key], fontsize=11)
        axes[idx].set_title(f'{metric_names[metric_key]} Statistics by Layer', fontsize=12)
        axes[idx].legend(loc='best', fontsize=9)
        axes[idx].grid(True, alpha=0.3)
    
    prompt = influence_dict[prompt_idx]['prompt']
    fig.suptitle(f'Layer-wise Statistics: "{prompt[:60]}..."', fontsize=14)
    plt.tight_layout()
    plt.show()

analyze_layer_statistics(influence_results, prompt_idx=0)

### Compare Influence Across Different Prompts

In [ ]:
def compare_prompts_influence(influence_dict, metric_key='frobenius_norm'):
    """
    Compare a specific metric across different prompts.
    """
    num_prompts = len(influence_dict)
    fig, axes = plt.subplots(1, num_prompts, figsize=(6*num_prompts, 5))
    
    if num_prompts == 1:
        axes = [axes]
    
    metric_names = {
        'frobenius_norm': 'Frobenius Norm',
        'spectral_norm': 'Spectral Norm',
        'participation_ratio': 'Participation Ratio',
        'entropy_effective_rank': 'Entropy Effective Rank'
    }
    
    for prompt_idx in range(num_prompts):
        prompt = influence_dict[prompt_idx]['prompt']
        tokens = tokenizer(prompt, return_tensors='pt')
        token_strs = [tokenizer.decode([tid]) for tid in tokens.input_ids[0]]
        
        data = influence_dict[prompt_idx][metric_key].cpu().numpy()
        
        im = axes[prompt_idx].imshow(data, cmap='viridis', aspect='auto')
        axes[prompt_idx].set_xlabel('Input Token', fontsize=10)
        axes[prompt_idx].set_ylabel('Layer', fontsize=10)
        axes[prompt_idx].set_title(f'Prompt {prompt_idx}\n"{prompt[:40]}..."', fontsize=11)
        axes[prompt_idx].set_xticks(range(len(token_strs)))
        axes[prompt_idx].set_xticklabels(token_strs, rotation=45, ha='right', fontsize=8)
        plt.colorbar(im, ax=axes[prompt_idx])
    
    fig.suptitle(f'{metric_names[metric_key]} Comparison', fontsize=14, y=1.00)
    plt.tight_layout()
    plt.show()

compare_prompts_influence(influence_results, metric_key='frobenius_norm')

## 6. Summary and Interpretation

### Block Coupling Metrics
- **Lower values** = stronger alignment between layer Jacobians
- **Diagonal** shows self-coupling (should be near zero)
- **Off-diagonal** patterns reveal how layers share similar computational structure
- **K parameter** controls how many singular vectors are considered

### Cross-Token Influence Metrics
1. **Frobenius Norm**: Overall magnitude of influence (√Σ S²)
   - Higher = token has stronger overall influence on output
   - Captures total effect across all directions

2. **Spectral Norm**: Maximum directional influence (max singular value)
   - Higher = token has strong influence in at least one direction
   - Most sensitive to dominant effects

3. **Participation Ratio**: How evenly influence is distributed across dimensions
   - Higher = influence spread across many dimensions
   - Lower = influence concentrated in few dimensions

4. **Entropy Effective Rank**: Another measure of influence distribution (entropy-based)
   - Similar to participation ratio but uses entropy
   - Higher = more uniform distribution of influence

### Expected Patterns
- **Recent tokens** often show stronger influence (positional bias)
- **Content words** may show higher influence than function words
- **Deeper layers** may show different influence patterns than early layers
- **Participation ratio** indicates if influence is concentrated or spread out

## 7. Save Results (Optional)

In [ ]:
# Optionally save results
save_results = False  # Set to True to save

if save_results:
    # Save to current directory or Google Drive if mounted
    output_dir = "./results"  # Change to save_dir if Drive is mounted
    os.makedirs(output_dir, exist_ok=True)
    
    # Save coupling results
    coupling_file = os.path.join(output_dir, f"{model_name}_coupling_results.pt")
    torch.save(coupling_results, coupling_file)
    print(f"✓ Saved coupling results to {coupling_file}")
    
    # Save influence results
    influence_file = os.path.join(output_dir, f"{model_name}_influence_results.pt")
    torch.save(influence_results, influence_file)
    print(f"✓ Saved influence results to {influence_file}")
    
    # If using Google Drive, files will persist after session ends
else:
    print("Results not saved (set save_results=True to save)")

## Next Steps

**Try different models:**
- Smaller models: `gpt2`, `facebook/opt-1.3b`, `EleutherAI/pythia-1.4b`
- Larger models: `meta-llama/Meta-Llama-3-8B` (requires HF auth)

**Try different prompts:**
- Longer sequences to see how influence patterns change
- Different types of text (code, poetry, questions)
- Multilingual text

**Analyze patterns:**
- Compare coupling between early vs late layers
- Look for attention-like patterns in influence metrics
- Correlate metrics with model performance

**Further reading:**
- Paper: [Transformer Block Coupling and its Correlation with Generalization in LLMs](https://arxiv.org/abs/2407.07810)
- Project page: https://sugolov.github.io/coupling/
- GitHub: https://github.com/sugolov/coupling